# Tutorial 14 — Graph Neural Networks for Molecules
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

Molecules are natural graphs: atoms are nodes, bonds are edges. GNNs exploit this structure directly — unlike fingerprints, they learn their own features end-to-end. We build a Graph Convolutional Network (GCN) using PyTorch Geometric.

In [ ]:
!pip install torch -q
!pip install torch-geometric -q
!pip install rdkit pandas numpy matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
from rdkit import Chem
import numpy as np

def mol_to_graph(smiles: str, label: float) -> Data | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    # Node features: atomic num, degree, formal charge, aromaticity, Hs
    x = []
    for atom in mol.GetAtoms():
        x.append([
            atom.GetAtomicNum() / 100.0,
            atom.GetDegree() / 6.0,
            atom.GetFormalCharge() / 2.0,
            float(atom.GetIsAromatic()),
            atom.GetTotalNumHs() / 4.0,
            float(atom.IsInRing()),
        ])
    x = torch.tensor(x, dtype=torch.float)
    # Edge index
    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index += [[i,j],[j,i]]
    if not edge_index:
        edge_index = torch.zeros((2,0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index, y=torch.tensor([label], dtype=torch.float))

# Build small dataset
dataset_raw = [
    ("CC(=O)Oc1ccccc1C(=O)O", 0),   # Aspirin - not hERG
    ("CC(C)Cc1ccc(cc1)C(C)C(=O)O", 0), # Ibuprofen - not hERG
    ("Cn1cnc2c1c(=O)n(C)c(=O)n2C", 0), # Caffeine - not hERG
    ("NCCc1ccc(O)c(O)c1", 0),           # Dopamine - not hERG
    ("CC(N)Cc1ccccc1", 0),              # Amphetamine - not hERG
    ("OC(c1ccc(C(c2ccccc2)(c2ccccc2)O)cc1)CCCN1CCC(CC1)C(O)(c1ccccc1)c1ccccc1", 1),  # Terfenadine - hERG
    ("Clc1ccc2c(c1)n(CCN1CCC(=C3c4cc(F)ccc4NC3=O)CC1)c(=O)n2", 1),  # Sertindole - hERG
    ("CN(CCOc1ccc(NS(=O)(=O)c2ccc(NC)cc2)cc1)S(=O)(=O)c1ccc(N)cc1", 1), # Dofetilide - hERG
]

graphs = [mol_to_graph(s, l) for s, l in dataset_raw]
graphs = [g for g in graphs if g is not None]
train_data = graphs[:6]; test_data = graphs[6:]

train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=2)
print(f"Dataset: {len(train_data)} train, {len(test_data)} test graphs")
print(f"Node features: {graphs[0].x.shape[1]}")

In [ ]:
class MolGCN(nn.Module):
    def __init__(self, in_ch=6, hidden=32, out_ch=1):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.fc    = nn.Linear(hidden, out_ch)
        self.drop  = nn.Dropout(0.3)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.drop(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        return torch.sigmoid(self.fc(x)).squeeze()

model = MolGCN()
opt   = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-4)
loss_fn = nn.BCELoss()

losses = []
for epoch in range(60):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        opt.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(pred, batch.y.squeeze())
        loss.backward(); opt.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(train_loader))
    if epoch % 15 == 0:
        print(f"Epoch {epoch:3d}  loss={losses[-1]:.4f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(7,4))
plt.plot(losses, color="#1565c0", lw=2)
plt.xlabel("Epoch"); plt.ylabel("BCE Loss"); plt.title("GCN Training Loss")
plt.tight_layout(); plt.savefig("gcn_loss.png", dpi=150); plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    for batch in test_loader:
        preds = model(batch.x, batch.edge_index, batch.batch)
        for p, t in zip(preds, batch.y):
            label = "hERG blocker" if t.item() == 1 else "non-blocker"
            pred_label = "hERG blocker" if p.item() > 0.5 else "non-blocker"
            print(f"  True: {label:15s}  Pred: {pred_label:15s}  Score: {p.item():.3f}")

## Key takeaways
- GNNs process molecules directly as graphs — no fingerprint featurization needed
- `global_mean_pool` aggregates node embeddings into a fixed-size graph embedding
- Small datasets (< 500 compounds) often don't outperform RF — GNNs shine with > 5k compounds
- Use pre-trained GNNs (e.g., from DeepChem or MolBERT) for small datasets via transfer learning